In [2]:
# Load env variables and create client
import os
from dotenv import load_dotenv
from anthropic import Anthropic
from statistics import mean

load_dotenv()

auth_token = os.getenv("ANTHROPIC_API_KEY")

client = Anthropic(
    auth_token=auth_token,
    base_url="https://flow.ciandt.com/flow-llm-proxy/"
)

model = "bedrock/anthropic.claude-4-6-sonnet"

In [3]:
# Helper functions
from anthropic.types import Message

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [ ]:
messages = []

add_user_message(messages, "Write a one paragraph guide to recursion.")

chat(messages, thinking=True)

Message(id='msg_018y4f7ZTmL54aVVGQ3CwJTn', container=None, content=[ThinkingBlock(signature='EvkBCmUIDhgCKkBRf5UA5Z8Y+hVCfRDsdBoAVkEIIwE0iB9eJ6712tvwKbWtCBWWsx3KkHBmrNkCGbOhIhq075dpaoA+OiQxoOV8MhFjbGF1ZGUtc29ubmV0LTQtNjgAQgh0aGlua2luZxIMpRXtPRwVcTS76WKMGgyzxD80PbH8VSzfVBoiMEPuyb11dJGSmexWmegPVqVoPL1Mfi1kiXFO72duviTx55x0Fr7zSI6BjIBssv+uHipCZXAdCfmRTyn3Ev1b14S1O0gE2wosKQdxyM5jcTLUdTySsScdXhqLzDMoKZBDGMWa63dPz0pC1sTvUIBzz8AsGG4EGAE=', thinking='The user wants a one paragraph guide to recursion.', type='thinking'), TextBlock(citations=None, text="## A Guide to Recursion\n\nRecursion is a programming technique where a function calls itself in order to solve a problem by breaking it down into smaller, simpler versions of the same problem. The key to writing a recursive function is defining two things: a **base case**, which is the stopping condition that returns a result without making another recursive call, and a **recursive case**, which breaks the problem down and calls the function agai